# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()
print("\nDATASET OVERVIEW:\n")
print(f"{metadata_json.get('name', 'No Name')}: {metadata_json.get('description', 'No Description')}")
print(f"Published: {metadata_json.get('datePublished', 'Unknown')}")
print(f"License: {metadata_json.get('license', 'Unknown')}")
print(f"Keywords: {', '.join(metadata_json.get('keywords', []))}")


## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

In [ ]:
# List out record sets and their fields by @id

record_sets = []  # Will be populated by @id
fields_by_record_set = {}

for rec in dataset.metadata.record_sets:
    record_sets.append(rec['@id'])
    fields = []
    for f in rec.get('fields', []):
        fields.append(f['@id'])
    fields_by_record_set[rec['@id']] = fields

print("\nRecord sets in dataset:")
for rs_id in record_sets:
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Fields (@id): {fields_by_record_set[rs_id]}")

# Optionally, visualize a sample record from each set
print("\nSample record(s) from each record set:")
for rs_id in record_sets:
    try:
        for idx, rec in enumerate(dataset.records(record_set=rs_id)):
            print(f"\n[{rs_id}] Sample record {idx+1}: {rec}")
            if idx == 0: break
    except Exception as e:
        print(f"Could not load records from {rs_id}: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All entities are referenced using their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
print("\nLoading records from all RecordSets...")

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecordSet @id: {record_set_id}")
    print(f"Fields (@id): {df.columns.tolist()}")
    print(df.head(2))  # Show first two records as preview


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, grouping by key attributes.

In [ ]:
# Choose a record set, numeric field, and group field by @id
# Inspect what fields are available, pick one for demonstration
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    # Find a numeric field (e.g., 'gad7_score', 'age', etc.)
    numeric_field_candidates = [col for col in df.columns if 'score' in col or 'age' in col or 'phq9' in col or 'psq' in col]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        print(f"\nUsing numeric field: {numeric_field}")
    else:
        numeric_field = df.columns[0]  # fallback

    # For demonstration, threshold and normalize
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    print(f"\nThreshold for filtering: {threshold}")

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field}:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field (e.g. 'gender', 'level_of_education')
    group_field_candidates = [col for col in df.columns if 'gender' in col or 'education' in col or 'village' in col or 'marital' in col]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"\nGrouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped means by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions of key numeric scores and relationships between demographic variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Distribution of a numeric field
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    numeric_field_candidates = [col for col in df.columns if 'score' in col or 'age' in col or 'phq9' in col or 'psq' in col]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()

    # Relationship: numeric vs. group field
    group_field_candidates = [col for col in df.columns if 'gender' in col or 'education' in col or 'village' in col or 'marital' in col]
    if numeric_field_candidates and group_field_candidates:
        group_field = group_field_candidates[0]
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This exploration demonstrated:
- Loading dataset metadata and records using the `mlcroissant` library.
- Referencing entities by their `@id` for clarity and consistency.
- Extracting all available record sets and fields.
- Performing basic data filtering, normalization, and grouping.
- Visualizing numeric data distributions and category group comparisons.

This FAIR² Mental Health Survey Dataset provides a strong foundation for further AI-driven analysis on mental health indicators and is structured to support reproducible and interoperable research.